In [ ]:
# ---- 1️⃣ Mount Google Drive cleanly ----
from google.colab import drive
import os, cv2, json, torch, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ---- 2️⃣ Create pipeline output directory structure ----
base_dir = "/content/drive/MyDrive/Store/pipeline_outputs/layer1"
os.makedirs(base_dir, exist_ok=True)
print("✅ Directory created:", base_dir)

✅ Directory created: /content/drive/MyDrive/Store/pipeline_outputs/layer1


In [ ]:
# ---- 3️⃣ Install all required packages ----
!pip install -q ultralytics transformers ensemble_boxes supervision==0.25.0 opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.2 MB/s eta 0:00:00


In [ ]:
# ---- 4️⃣ Import required libraries ----
from ultralytics import YOLO
from transformers import AutoImageProcessor, DeformableDetrForObjectDetection
from ensemble_boxes import weighted_boxes_fusion
import supervision as sv

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# ---- 5️⃣ Define the run_layer1() function ----
def run_layer1(input_path, yolo_model_path,
               save_root="/content/drive/MyDrive/Store/pipeline_outputs"):

    save_root = Path(save_root)
    layer1_root = save_root / "layer1"
    yolo_out = layer1_root / "yolo_dets"
    deform_detr_out = layer1_root / "deform_detr_dets"
    fused_out = layer1_root / "fused_dets"
    track_out = layer1_root / "byte_tracks"
    feature_out = layer1_root / "frame_features"
    flow_ready_frames = layer1_root / "frames_for_farneback"

    for d in [yolo_out, deform_detr_out, fused_out, track_out, feature_out, flow_ready_frames]:
        d.mkdir(parents=True, exist_ok=True)

    def extract_frames(video_path, out_dir):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        cap = cv2.VideoCapture(str(video_path))
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            cv2.imwrite(str(out_dir / f"{idx:06d}.jpg"), frame)
            idx += 1
        cap.release()
        return sorted(out_dir.glob("*.jpg"))

    input_path = Path(input_path)
    if input_path.is_file() and input_path.suffix.lower() in [".mp4", ".avi", ".mov", ".mkv"]:
        img_paths = extract_frames(input_path, flow_ready_frames)
    elif input_path.is_dir():
        img_paths = sorted(list(input_path.glob("*.jpg")) + list(input_path.glob("*.png")))
    else:
        raise ValueError(f"Invalid input path: {input_path}")

    yolo = YOLO(yolo_model_path)
    deform_processor = AutoImageProcessor.from_pretrained("SenseTime/deformable-detr")
    deform_model = DeformableDetrForObjectDetection.from_pretrained("SenseTime/deformable-detr")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    deform_model.to(device).eval()
    tracker = sv.ByteTrack()

    def normalize_boxes(boxes, width, height):
        return [[x1/width, y1/height, x2/width, y2/height] for x1, y1, x2, y2, _, _ in boxes]

    summary_rows = []

    for img_path in img_paths:
        im = Image.open(img_path).convert("RGB")
        W, H = im.width, im.height

        # YOLO detections
        res = yolo.predict(str(img_path), imgsz=640, conf=0.25, verbose=False)[0]
        yolo_dets = []
        if res.boxes is not None:
            for box in res.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = float(box.conf)
                cls = int(box.cls)
                if cls == 0:
                    yolo_dets.append([x1, y1, x2, y2, conf, cls])
        with open(yolo_out / f"{img_path.stem}.json", "w") as f:
            json.dump(yolo_dets, f)

        # Deformable DETR detections
        inputs = deform_processor(images=im, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = deform_model(**inputs)
        results = deform_processor.post_process_object_detection(
            outputs, target_sizes=torch.tensor([[H, W]], device=device), threshold=0.25
        )[0]

        deform_dets = []
        for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
            if int(label) == 1:
                x1, y1, x2, y2 = box.tolist()
                deform_dets.append([float(x1), float(y1), float(x2), float(y2), float(score), 0])
        with open(deform_detr_out / f"{img_path.stem}.json", "w") as f:
            json.dump(deform_dets, f)

        # Weighted Boxes Fusion
        all_boxes, all_scores, all_labels = [], [], []
        if len(yolo_dets) > 0:
            all_boxes.append(normalize_boxes(yolo_dets, W, H))
            all_scores.append([d[4] for d in yolo_dets])
            all_labels.append([d[5] for d in yolo_dets])
        if len(deform_dets) > 0:
            all_boxes.append(normalize_boxes(deform_dets, W, H))
            all_scores.append([d[4] for d in deform_dets])
            all_labels.append([d[5] for d in deform_dets])

        fused_dets = []
        if len(all_boxes) > 0:
            fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
                all_boxes, all_scores, all_labels, iou_thr=0.5, skip_box_thr=0.05
            )
            for box, score, label in zip(fused_boxes, fused_scores, fused_labels):
                x1, y1, x2, y2 = box[0]*W, box[1]*H, box[2]*W, box[3]*H
                fused_dets.append([x1, y1, x2, y2, float(score), int(label)])
        with open(fused_out / f"{img_path.stem}.json", "w") as f:
            json.dump(fused_dets, f)

        # ByteTrack tracking
        dets_array = np.empty((0, 6))
        if len(fused_dets) > 0:
            xyxy = np.array([d[:4] for d in fused_dets], dtype=np.float32)
            conf = np.array([d[4] for d in fused_dets], dtype=np.float32)
            cls_id = np.array([d[5] for d in fused_dets], dtype=np.int32)
            detections = sv.Detections(xyxy=xyxy, confidence=conf, class_id=cls_id)
            tracks = tracker.update_with_detections(detections)
            if len(tracks) > 0:
                dets_array = np.hstack([
                    tracks.xyxy,
                    tracks.confidence[:, None],
                    tracks.tracker_id[:, None]
                ])
        np.save(track_out / f"{img_path.stem}.npy", dets_array)

        # Frame-level features
        if len(dets_array) > 0:
            num_people = len(dets_array)
            areas = (dets_array[:, 2] - dets_array[:, 0]) * (dets_array[:, 3] - dets_array[:, 1])
            mean_area = float(areas.mean()) if areas.size else 0.0
            mean_conf = float(dets_array[:, 4].mean())
        else:
            num_people, mean_area, mean_conf = 0, 0.0, 0.0

        np.save(feature_out / f"{img_path.stem}_feat.npy",
                np.array([num_people, mean_area, mean_conf], dtype=np.float32))

        summary_rows.append({
            "frame": img_path.name,
            "num_people": num_people,
            "mean_area": mean_area,
            "mean_conf": mean_conf
        })

        im.close()

    pd.DataFrame(summary_rows).to_csv(layer1_root / "layer1_summary.csv", index=False)

    return {
        "layer1_root": str(layer1_root),
        "yolo_out": str(yolo_out),
        "deform_detr_out": str(deform_detr_out),
        "fused_out": str(fused_out),
        "track_out": str(track_out),
        "feature_out": str(feature_out),
        "flow_ready_frames": str(flow_ready_frames),
        "summary_csv": str(layer1_root / "layer1_summary.csv")
    }


# ---- 6️⃣ Run Layer 1 ----
layer1_results = run_layer1(
    input_path="/content/drive/MyDrive/Store/MOT20/test/MOT20-07/img1",
    yolo_model_path="/content/drive/MyDrive/Store/L1model/yolo_train/yolov8n_mot20_gpu_headft_smallimg/weights/best.pt"
)

print("✅ Layer1 completed! Outputs saved to:")
print(layer1_results)

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


preprocessor_config.json:   0%|          | 0.00/305 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/161M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Some weights of the model checkpoint at SenseTime/deformable-detr were not used when initializing DeformableDetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DeformableDetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DeformableDetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✅ Layer1 completed! Outputs saved to:
{'layer1_root': '/content/drive/MyDrive/Store/pipeline_outputs/layer1', 'yolo_out': '/content/drive/MyDrive/Store/pipeline_outputs/layer1/yolo_dets', 'deform_detr_out': '/content/drive/MyDrive/Store/pipeline_outputs/layer1/deform_detr_dets', 'fused_out': '/content/drive/MyDrive/Store/pipeline_outputs/layer1/fused_dets', 'track_out': '/content/drive/MyDrive/Store/pipeline_outputs/layer1/byte_tracks', 'feature_out': '/content/drive/MyDrive/Store/pipeline_outputs/layer1/frame_features', 'flow_ready_frames': '/content/drive/MyDrive/Store/pipeline_outputs/layer1/frames_for_farneback', 'summary_csv': '/content/drive/MyDrive/Store/pipeline_outputs/layer1/layer1_summary.csv'}


### LAYER 1

In [ ]:
# ===============================================================
#  LAYER 1 PIPELINE — YOLO + Deformable DETR + Fusion + ByteTrack
#  Automatically saves frames for Farneback (Layer 2)
# ===============================================================

import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from transformers import DeformableDetrForObjectDetection, DeformableDetrImageProcessor
from ensemble_boxes import weighted_boxes_fusion
from tqdm import tqdm
from pathlib import Path
import torch

# ---------------------------------------------------------------
# Function to ensure Drive directories exist
# ---------------------------------------------------------------
def ensure_dirs(base_dir):
    subdirs = ["yolo_dets", "deform_detr_dets", "fused_dets",
               "byte_tracks", "frame_features", "frames_for_farneback"]
    for s in subdirs:
        os.makedirs(os.path.join(base_dir, s), exist_ok=True)
    return {s: os.path.join(base_dir, s) for s in subdirs}

# ---------------------------------------------------------------
# Main Layer 1 function
# ---------------------------------------------------------------
def run_layer1(input_path, yolo_model_path, save_root="/content/drive/MyDrive/Store/pipeline_outputs/layer1"):
    # ✅ Create output directories
    dirs = ensure_dirs(save_root)

    # ✅ Collect frames (video or folder)
    input_path = Path(input_path)
    if input_path.is_file():
        cap = cv2.VideoCapture(str(input_path))
        frame_paths = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
            frame_path = dirs["frames_for_farneback"] + f"/frame_{frame_idx:05d}.jpg"
            cv2.imwrite(frame_path, frame)
            frame_paths.append(frame_path)
        cap.release()
    elif input_path.is_dir():
        frame_paths = sorted([str(p) for p in input_path.glob("*.jpg")])
        # 💡 Copy or save frames for Farneback
        for p in tqdm(frame_paths, desc="Saving frames for Farneback"):
            img = cv2.imread(p)
            fname = os.path.basename(p)
            cv2.imwrite(os.path.join(dirs["frames_for_farneback"], fname), img)
    else:
        raise ValueError("Input must be a video file or a folder of frames")

    # ✅ Load YOLO and Deformable DETR
    yolo_model = YOLO(yolo_model_path)
    detr_processor = DeformableDetrImageProcessor.from_pretrained("SenseTime/deformable-detr")
    detr_model = DeformableDetrForObjectDetection.from_pretrained("SenseTime/deformable-detr")

    all_summary = []

    for frame_path in tqdm(frame_paths, desc="Running Layer 1"):
        frame = cv2.imread(frame_path)
        h, w, _ = frame.shape

        # YOLOv8 detection
        yolo_results = yolo_model(frame)[0]
        y_boxes = yolo_results.boxes.xyxy.cpu().numpy() / np.array([w, h, w, h])
        y_scores = yolo_results.boxes.conf.cpu().numpy()

        # Save YOLO detections
        if len(y_boxes) > 0:
             y_df = pd.DataFrame(np.hstack((y_boxes, y_scores[:, None])),
                        columns=["x1","y1","x2","y2","score"])
        else:
            y_df = pd.DataFrame(columns=["x1","y1","x2","y2","score"])
        y_df.to_csv(os.path.join(dirs["yolo_dets"], Path(frame_path).stem + "_yolo.csv"), index=False)

        # Deformable DETR detection
        inputs = detr_processor(images=frame, return_tensors="pt")
        with torch.no_grad():
            outputs = detr_model(**inputs)
        target_sizes = torch.tensor([frame.shape[:2]])
        results = detr_processor.post_process_object_detection(outputs, target_sizes=target_sizes)[0]
        d_boxes = np.array([[b[0]/w, b[1]/h, b[2]/w, b[3]/h] for b in results["boxes"]])
        d_scores = np.array(results["scores"])

        # Save DETR detections
        if len(d_boxes) > 0:
              d_df = pd.DataFrame(np.hstack((d_boxes, d_scores[:, None])),
                        columns=["x1","y1","x2","y2","score"])
        else:
              d_df = pd.DataFrame(columns=["x1","y1","x2","y2","score"])

        d_df.to_csv(os.path.join(dirs["deform_detr_dets"], Path(frame_path).stem + "_detr.csv"), index=False)

        # Fuse detections
        if len(y_boxes) and len(d_boxes):
            dummy_labels = [[0]*len(y_boxes), [0]*len(d_boxes)]
            fused_boxes, fused_scores, _ = weighted_boxes_fusion(
                  [y_boxes, d_boxes], [y_scores, d_scores], dummy_labels, weights=[2, 1], iou_thr=0.55
            )

        elif len(y_boxes):
            fused_boxes, fused_scores = y_boxes, y_scores
        else:
            fused_boxes, fused_scores = d_boxes, d_scores

        # Save fused results
        f_df = pd.DataFrame(np.hstack((fused_boxes, fused_scores[:, None])),
                            columns=["x1","y1","x2","y2","score"])
        f_df.to_csv(os.path.join(dirs["fused_dets"], Path(frame_path).stem + "_fused.csv"), index=False)

        # Feature summary
        num_people = len(fused_boxes)
        mean_area = np.mean((fused_boxes[:,2]-fused_boxes[:,0])*(fused_boxes[:,3]-fused_boxes[:,1])) if num_people > 0 else 0
        mean_conf = np.mean(fused_scores) if num_people > 0 else 0

        all_summary.append({
            "frame": Path(frame_path).stem,
            "num_people": num_people,
            "mean_area": mean_area,
            "mean_conf": mean_conf
        })

    # Save overall summary CSV
    df_summary = pd.DataFrame(all_summary)
    csv_path = os.path.join(save_root, "layer1_summary.csv")
    df_summary.to_csv(csv_path, index=False)
    print(f"✅ Layer 1 summary saved to: {csv_path}")

    return {
        "summary_csv": csv_path,
        "frames_for_farneback": dirs["frames_for_farneback"]
    }

In [ ]:
!ls -lh "/content/drive/MyDrive/Store/pipeline_outputs/layer1"

total 55K
drwx------ 2 root root 4.0K Nov 10 16:00 byte_tracks
drwx------ 2 root root 4.0K Nov 10 16:00 deform_detr_dets
drwx------ 2 root root 4.0K Nov 10 16:00 frame_features
drwx------ 2 root root 4.0K Nov 10 15:54 frames_for_farneback
drwx------ 2 root root 4.0K Nov 10 16:00 fused_dets
-rw------- 1 root root  31K Nov 10 16:00 layer1_summary.csv
drwx------ 2 root root 4.0K Nov 10 16:00 yolo_dets


In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/Store/MOT20/test/MOT20-07/img1")
print("Exists:", p.exists())
print("Is file:", p.is_file())
print("Is directory:", p.is_dir())

!ls -lh "/content/drive/MyDrive/Store/MOT20/test/MOT20-07/img1" | head

Exists: True
Is file: False
Is directory: True
total 245M
-rw------- 1 root root 375K Feb 28  2020 000001.jpg
-rw------- 1 root root 363K Feb 28  2020 000002.jpg
-rw------- 1 root root 363K Feb 28  2020 000003.jpg
-rw------- 1 root root 372K Feb 28  2020 000004.jpg
-rw------- 1 root root 380K Feb 28  2020 000005.jpg
-rw------- 1 root root 383K Feb 28  2020 000006.jpg
-rw------- 1 root root 382K Feb 28  2020 000007.jpg
-rw------- 1 root root 392K Feb 28  2020 000008.jpg
-rw------- 1 root root 391K Feb 28  2020 000009.jpg


In [ ]:
import os

layer1_dir = "/content/drive/MyDrive/Store/pipeline_outputs/layer1"
frames_dir = os.path.join(layer1_dir, "frames_for_farneback")

# make sure the folder exists
os.makedirs(frames_dir, exist_ok=True)
print("✅ Folder check complete:")
!ls -lh "$layer1_dir"

✅ Folder check complete:
total 55K
drwx------ 2 root root 4.0K Nov 10 16:00 byte_tracks
drwx------ 2 root root 4.0K Nov 10 16:00 deform_detr_dets
drwx------ 2 root root 4.0K Nov 10 16:00 frame_features
drwx------ 2 root root 4.0K Nov 10 15:54 frames_for_farneback
drwx------ 2 root root 4.0K Nov 10 16:00 fused_dets
-rw------- 1 root root  31K Nov 10 16:00 layer1_summary.csv
drwx------ 2 root root 4.0K Nov 10 16:00 yolo_dets


In [ ]:
layer1_results = run_layer1(
    input_path="/content/drive/MyDrive/Store/MOT20/test/MOT20-07/img1",
    yolo_model_path="/content/drive/MyDrive/Store/L1model/yolo_train/yolov8n_mot20_gpu_headft_smallimg/weights/best.pt"
)

Saving frames for Farneback: 100%|██████████| 595/595 [00:27<00:00, 21.80it/s]
Some weights of the model checkpoint at SenseTime/deformable-detr were not used when initializing DeformableDetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DeformableDetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DeformableDetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequ


0: 192x320 119 persons, 8.7ms
Speed: 1.0ms preprocess, 8.7ms inference, 1.4ms postprocess per image at shape (1, 3, 192, 320)


Running Layer 1:   0%|          | 1/595 [00:19<3:11:04, 19.30s/it]


0: 192x320 126 persons, 6.5ms
Speed: 1.0ms preprocess, 6.5ms inference, 1.5ms postprocess per image at shape (1, 3, 192, 320)


Running Layer 1:   0%|          | 2/595 [00:39<3:17:36, 19.99s/it]

...


0: 192x320 62 persons, 9.9ms
Speed: 1.1ms preprocess, 9.9ms inference, 1.8ms postprocess per image at shape (1, 3, 192, 320)


Running Layer 1: 100%|█████████▉| 594/595 [3:10:36<00:19, 19.43s/it]


0: 192x320 63 persons, 8.6ms
Speed: 1.1ms preprocess, 8.6ms inference, 1.4ms postprocess per image at shape (1, 3, 192, 320)


Running Layer 1: 100%|██████████| 595/595 [3:10:56<00:00, 19.25s/it]

✅ Layer 1 summary saved to: /content/drive/MyDrive/Store/pipeline_outputs/layer1/layer1_summary.csv


In [ ]:
!ls -lh "/content/drive/MyDrive/Store/pipeline_outputs/layer1/frames_for_farneback" | head

total 269M
-rw------- 1 root root 417K Nov 10 16:22 000001.jpg
-rw------- 1 root root 407K Nov 10 16:22 000002.jpg
-rw------- 1 root root 406K Nov 10 16:22 000003.jpg
-rw------- 1 root root 416K Nov 10 16:22 000004.jpg
-rw------- 1 root root 422K Nov 10 16:22 000005.jpg
-rw------- 1 root root 427K Nov 10 16:22 000006.jpg
-rw------- 1 root root 425K Nov 10 16:22 000007.jpg
-rw------- 1 root root 435K Nov 10 16:22 000008.jpg
-rw------- 1 root root 433K Nov 10 16:22 000009.jpg


# LAYER 2

In [ ]:
!pip install -q tensorflow

In [ ]:
# ============================
# Layer 2 — Farneback + Heuristic + Dual-LSTM
# Saves everything under /content/drive/MyDrive/Store/pipeline_outputs/layer2/
# ============================

import os
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm

# ----------------------------
# Farneback feature extractor
# ----------------------------
def flow_features(prev_gray, gray, grid=(16,16), mag_thresh=0.8):
    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None,
                                        0.5, 3, 15, 3, 5, 1.2, 0)
    fx, fy = flow[..., 0], flow[..., 1]
    mag, ang = cv2.cartToPolar(fx, fy, angleInDegrees=False)

    active = (mag > mag_thresh).astype(np.uint8)
    density = float(active.mean())

    mag_mean = float(mag.mean())
    mag_std = float(mag.std())

    hist, _ = np.histogram(mag, bins=32, range=(0, np.clip(mag.max(), 1e-6, 10)))
    p = hist / (hist.sum() + 1e-12)
    entropy = float((-(p[p>0] * np.log(p[p>0]))).sum())

    ux, uy = np.cos(ang), np.sin(ang)
    R = np.sqrt(ux.mean()**2 + uy.mean()**2)
    dir_var = float(1 - R)

    pressure = float(density * mag_mean)

    # Heatmap grid
    H, W = mag.shape
    gh, gw = grid
    cell_h, cell_w = max(1, H // gh), max(1, W // gw)
    heat = np.zeros((gh, gw), dtype=np.float32)
    for i in range(gh):
        for j in range(gw):
            y0, y1 = i * cell_h, (i + 1) * cell_h
            x0, x1 = j * cell_w, (j + 1) * cell_w
            # clamp to image bounds
            y1 = min(y1, H); x1 = min(x1, W)
            cell = active[y0:y1, x0:x1]
            heat[i, j] = float(cell.mean()) if cell.size > 0 else 0.0

    feat_vec = np.array([density, mag_mean, mag_std, entropy, dir_var, pressure], dtype=np.float32)
    return feat_vec, heat

In [ ]:
# ----------------------------
# Process sequence -> .npz + CSV
# ----------------------------
def process_image_sequence(img_dir,
                           grid=(16,16),
                           out_dir="/content/drive/MyDrive/Store/pipeline_outputs/layer2/Farneback_features"):
    os.makedirs(out_dir, exist_ok=True)
    img_files = sorted(glob(os.path.join(img_dir, "*.jpg")) + glob(os.path.join(img_dir, "*.png")))
    if len(img_files) < 2:
        raise ValueError(f"Not enough frames in {img_dir} (need >=2)")

    feat_list, heat_list = [], []
    prev_frame = cv2.imread(img_files[0])
    if prev_frame is None:
        raise ValueError(f"Could not read first frame: {img_files[0]}")
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    for img_path in tqdm(img_files[1:], desc="Optical Flow"):
        frame = cv2.imread(img_path)
        if frame is None:
            # skip unreadable frames
            prev_gray = prev_gray
            continue
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        feat_vec, heat = flow_features(prev_gray, gray, grid)
        feat_list.append(feat_vec)
        heat_list.append(heat)
        prev_gray = gray

    feat_arr = np.stack(feat_list) if len(feat_list) > 0 else np.zeros((0,6), dtype=np.float32)
    heat_arr = np.stack(heat_list) if len(heat_list) > 0 else np.zeros((0, grid[0], grid[1]), dtype=np.float32)

    npz_path = os.path.join(out_dir, "sequence_flow.npz")
    np.savez_compressed(npz_path, features=feat_arr, heatmaps=heat_arr)

    # CSV summary
    if feat_arr.shape[0] > 0:
        df = pd.DataFrame(feat_arr, columns=["density","mag_mean","mag_std","entropy","dir_var","pressure"])
    else:
        df = pd.DataFrame(columns=["density","mag_mean","mag_std","entropy","dir_var","pressure"])
    csv_path = os.path.join(out_dir, "flow_features_summary.csv")
    df.to_csv(csv_path, index=False)

    return {
        "npz": npz_path,
        "csv": csv_path,
        "features_shape": feat_arr.shape,
        "heat_shape": heat_arr.shape
    }

In [ ]:
# ----------------------------
# Heuristic labeling (and save)
# ----------------------------
def heuristic_labeling(features,
                        density_thresh=0.3,
                        speed_thresh=1.5,
                        entropy_thresh=1.0):
    labels = []
    for f in features:
        density, mag_mean, mag_std, entropy, dir_var, pressure = map(float, f)
        risk_score = 0
        if density > density_thresh:    risk_score += 1
        if mag_mean > speed_thresh:     risk_score += 1
        if entropy > entropy_thresh or dir_var > 0.5: risk_score += 1
        labels.append(1 if risk_score >= 2 else 0)
    return np.array(labels, dtype=np.int64)


def process_npz_file(npz_file,
                     out_dir="/content/drive/MyDrive/Store/pipeline_outputs/layer2/Heuristic_labels"):
    os.makedirs(out_dir, exist_ok=True)
    data = np.load(npz_file)
    features = data["features"]
    heatmaps = data["heatmaps"]

    labels = heuristic_labeling(features)

    labeled_path = os.path.join(out_dir, "sequence_labeled.npz")
    np.savez_compressed(labeled_path, features=features, heatmaps=heatmaps, labels=labels)

    # CSV summary
    if features.shape[0] > 0:
        df = pd.DataFrame(features, columns=["density","mag_mean","mag_std","entropy","dir_var","pressure"])
        df["label"] = labels
    else:
        df = pd.DataFrame(columns=["density","mag_mean","mag_std","entropy","dir_var","pressure","label"])
    csv_path = os.path.join(out_dir, "heuristic_labels_summary.csv")
    df.to_csv(csv_path, index=False)

    return {
        "labeled_npz": labeled_path,
        "csv": csv_path,
        "num_records": int(features.shape[0])
    }


In [ ]:
# ----------------------------
# Combined runner for Layer 2
# ----------------------------
def run_layer2(input_path,
               base_out="/content/drive/MyDrive/Store/pipeline_outputs/layer2"):
    """
    input_path: folder of frames (jpg/png)
    base_out: root folder for layer2 outputs (will be created)
    Returns dict of saved paths.
    """
    os.makedirs(base_out, exist_ok=True)
    farneback_out = os.path.join(base_out, "Farneback_features")
    heuristic_out = os.path.join(base_out, "Heuristic_labels")

    # Step 1: Farneback features
    farneback_result = process_image_sequence(input_path, out_dir=farneback_out)

    # Step 2: Heuristic labeling
    labeled_result = process_npz_file(farneback_result["npz"], out_dir=heuristic_out)

    # Layer-level summary CSV
    data = np.load(labeled_result["labeled_npz"])
    feats = data["features"]
    labels = data["labels"] if "labels" in data else np.zeros((feats.shape[0],), dtype=np.int64)
    if feats.shape[0] > 0:
        df_summary = pd.DataFrame(feats, columns=["density","mag_mean","mag_std","entropy","dir_var","pressure"])
        df_summary["label"] = labels
    else:
        df_summary = pd.DataFrame(columns=["density","mag_mean","mag_std","entropy","dir_var","pressure","label"])
    summary_csv = os.path.join(base_out, "layer2_summary.csv")
    df_summary.to_csv(summary_csv, index=False)

    return {
        "farneback_npz": farneback_result["npz"],
        "farneback_csv": farneback_result["csv"],
        "heuristic_npz": labeled_result["labeled_npz"],
        "heuristic_csv": labeled_result["csv"],
        "layer2_summary_csv": summary_csv
    }

In [ ]:
# ----------------------------
# Dual-LSTM inference (per-window) — SAVES CSVs
# ----------------------------
def run_layer2_lstm(npz_file,
                    save_root="/content/drive/MyDrive/Store/pipeline_outputs/layer2/lstm",
                    lstmMODEL_PATH="/content/drive/MyDrive/Store/models/model_w_kfold2/dual_branch_fold4.keras",
                    FEAT_MEAN_PATH="/content/drive/MyDrive/Store/models/model_w_kfold2/feat_mean_fold4.npy",
                    FEAT_STD_PATH="/content/drive/MyDrive/Store/models/model_w_kfold2/feat_std_fold4.npy",
                    BYTE_DIR="/content/detections/tracks",
                    WINDOW=32, STRIDE=8):
    """
    Loads the labeled npz and runs your keras model per-window.
    Saves:
      - lstm_window_probs.csv
      - lstm_summary.csv
    Returns dict of saved file paths and stats.
    """
    os.makedirs(save_root, exist_ok=True)
    out_probs_csv = os.path.join(save_root, "lstm_window_probs.csv")
    out_summary_csv = os.path.join(save_root, "lstm_summary.csv")

    # Import tensorflow lazily so the primary layer-2 flow doesn't require it unless used
    import tensorflow as tf

    # Load model + stats
    model = tf.keras.models.load_model(lstmMODEL_PATH)
    feat_mean = np.load(FEAT_MEAN_PATH)
    feat_std = np.load(FEAT_STD_PATH)

    # Helper to load ByteTrack features if present
    def load_bytetrack_features(npz_file, bytetrack_dir=BYTE_DIR):
        base = os.path.basename(npz_file).replace("_heuristic.npz", "").replace("sequence_labeled.npz", "")
        # attempt various suffix patterns
        candidates = [
            os.path.join(bytetrack_dir, base + "_bytetrack.npy"),
            os.path.join(bytetrack_dir, base + "_bytetrack.npz"),
            os.path.join(bytetrack_dir, base + ".npy"),
        ]
        for c in candidates:
            if os.path.exists(c):
                try:
                    return np.load(c)
                except:
                    pass
        return None

    data = np.load(npz_file)
    features = data["features"]
    heatmaps = data["heatmaps"]
    labels = data["labels"] if "labels" in data else None

    # Merge ByteTrack if shapes align
    bt_feats = load_bytetrack_features(npz_file)
    if bt_feats is not None and bt_feats.shape[0] == features.shape[0]:
        try:
            features = np.concatenate([features, bt_feats], axis=1)
        except:
            # If concatenation fails, ignore bt_feats
            pass

    # Windowed prediction
    probs = []
    T = features.shape[0]
    for start in range(0, max(T - WINDOW + 1, 0), STRIDE):
        end = start + WINDOW
        f_window = features[start:end].astype(np.float32)
        h_window = heatmaps[start:end].astype(np.float32)

        # Normalize features
        if feat_mean.shape == f_window.shape[1:]:
            # unlikely; fallback to elementwise
            f_window = (f_window - feat_mean) / (feat_std + 1e-8)
        else:
            f_window = (f_window - feat_mean) / (feat_std + 1e-8)

        # Add batch dims expected by your model
        f_in = np.expand_dims(f_window, 0)            # (1, W, F)
        h_in = np.expand_dims(h_window[..., None], 0) # (1, W, H, W, 1) or similar as earlier
        try:
            prob = model.predict({"features": f_in, "heatmaps": h_in}, verbose=0)[0,0]
        except:
            # fallback if model expects plain list/tuple
            prob = model.predict([f_in, h_in], verbose=0)[0,0]
        probs.append(float(prob))

    probs = np.array(probs, dtype=np.float32)

    # Save per-window CSV
    rows = []
    for i, p in enumerate(probs, 1):
        level = "Very Low" if p < 0.3 else "Low" if p < 0.5 else "High" if p < 0.8 else "Very High"
        rows.append({"window_id": int(i), "probability": float(p), "risk_level": level})
    if len(rows) > 0:
        pd.DataFrame(rows).to_csv(out_probs_csv, index=False)
    else:
        pd.DataFrame(columns=["window_id","probability","risk_level"]).to_csv(out_probs_csv, index=False)

    # Summary
    mean_prob = float(np.mean(probs)) if probs.size>0 else 0.0
    max_prob = float(np.max(probs)) if probs.size>0 else 0.0
    high_count = int(np.sum(probs > 0.8)) if probs.size>0 else 0
    low_count = int(np.sum(probs <= 0.5)) if probs.size>0 else 0
    summary_df = pd.DataFrame([{
        "num_windows": int(len(probs)),
        "mean_probability": mean_prob,
        "max_probability": max_prob,
        "high_risk_windows": high_count,
        "low_risk_windows": low_count
    }])
    summary_df.to_csv(out_summary_csv, index=False)

    return {
        "lstm_window_probs_csv": out_probs_csv,
        "lstm_summary_csv": out_summary_csv,
        "num_windows": int(len(probs)),
        "mean_probability": mean_prob,
        "max_probability": max_prob
    }

In [ ]:
layer2_results = run_layer2("/content/drive/MyDrive/Store/pipeline_outputs/layer1/frames_for_farneback")

Optical Flow: 100%|██████████| 594/594 [10:41<00:00,  1.08s/it]


In [ ]:
!ls -lh "/content/drive/MyDrive/Store/pipeline_outputs/layer2"

total 45K
drwx------ 2 root root 4.0K Nov 11 07:11 Farneback_features
drwx------ 2 root root 4.0K Nov 11 07:11 Heuristic_labels
-rw------- 1 root root  37K Nov 11 07:11 layer2_summary.csv


In [ ]:
!ls -lh "/content/drive/MyDrive/Store/pipeline_outputs/layer2/Farneback_features" | head
!ls -lh "/content/drive/MyDrive/Store/pipeline_outputs/layer2/Heuristic_labels" | head

total 437K
-rw------- 1 root root  36K Nov 11 07:11 flow_features_summary.csv
-rw------- 1 root root 401K Nov 11 07:11 sequence_flow.npz
total 438K
-rw------- 1 root root  37K Nov 11 07:11 heuristic_labels_summary.csv
-rw------- 1 root root 401K Nov 11 07:11 sequence_labeled.npz


In [ ]:
!head "/content/drive/MyDrive/Store/pipeline_outputs/layer2/layer2_summary.csv"

density,mag_mean,mag_std,entropy,dir_var,pressure,label
0.31611836,0.9912939,1.5820137,1.9877625,0.6059977,0.31336617,1
0.39031154,0.9906219,1.4193158,2.051286,0.63692474,0.38665116,1
0.45152488,1.0838088,1.4847032,2.1689303,0.6518682,0.48936662,1
0.610164,1.7553402,1.7535881,2.4860764,0.17879462,1.0710454,1
0.60212386,1.7196695,1.7998468,2.4911814,0.23702604,1.035454,1
0.5889906,1.4519838,1.609197,2.3856506,0.24198383,0.8552049,1
0.32271218,0.9695758,1.5254608,2.0026498,0.6863913,0.31289393,1
0.6007031,1.5698631,1.8454813,2.402854,0.2331183,0.94302166,1
0.43522522,1.20868,1.7625606,2.2497284,0.42866433,0.526048,1


In [ ]:
# ⚙️ Layer 2 — Dual LSTM Inference
lstm_results = run_layer2_lstm(
    npz_file="/content/drive/MyDrive/Store/pipeline_outputs/layer2/Heuristic_labels/sequence_labeled.npz"
)

lstm_results

{'lstm_window_probs_csv': '/content/drive/MyDrive/Store/pipeline_outputs/layer2/lstm/lstm_window_probs.csv',
 'lstm_summary_csv': '/content/drive/MyDrive/Store/pipeline_outputs/layer2/lstm/lstm_summary.csv',
 'num_windows': 71,
 'mean_probability': 0.8841128945350647,
 'max_probability': 0.9999393820762634}

In [ ]:
!ls -lh "/content/drive/MyDrive/Store/pipeline_outputs/layer2/lstm"

total 3.0K
-rw------- 1 root root  127 Nov 11 07:21 lstm_summary.csv
-rw------- 1 root root 2.2K Nov 11 07:21 lstm_window_probs.csv


In [ ]:
!head "/content/drive/MyDrive/Store/pipeline_outputs/layer2/lstm/lstm_window_probs.csv"
!cat "/content/drive/MyDrive/Store/pipeline_outputs/layer2/lstm/lstm_summary.csv"

window_id,probability,risk_level
1,0.9998904466629028,Very High
2,0.999832272529602,Very High
3,0.999928891658783,Very High
4,0.9998922348022461,Very High
5,0.9998860955238342,Very High
6,0.9999016523361206,Very High
7,0.9999152421951294,Very High
8,0.9999393820762634,Very High
9,0.9999291300773621,Very High
num_windows,mean_probability,max_probability,high_risk_windows,low_risk_windows
71,0.8841128945350647,0.9999393820762634,59,10


In [ ]:
!apt-get install tree -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 1s (68.2 kB/s)
Selecting previously unselected package tree.
(Reading database ... 121229 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!tree /content/drive/MyDrive/Store/pipeline_outputs/layer2 -L 2

/content/drive/MyDrive/Store/pipeline_outputs/layer2
├── Farneback_features
│   ├── flow_features_summary.csv
│   └── sequence_flow.npz
├── Heuristic_labels
│   ├── heuristic_labels_summary.csv
│   └── sequence_labeled.npz
├── layer2_summary.csv
└── lstm
    ├── lstm_summary.csv
    └── lstm_window_probs.csv

3 directories, 7 files


# LAYER 3

In [ ]:
import os, json, numpy as np, pandas as pd
from pathlib import Path
from tqdm import tqdm

# ==============================================================
# CONFIG
# ==============================================================
L1_TRACKS_DIR = Path("/content/drive/MyDrive/Store/pipeline_outputs/layer1/byte_tracks")
SAVE_ROOT = Path("/content/drive/MyDrive/Store/pipeline_outputs/layer3")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
print("✅ Layer3 directory:", SAVE_ROOT)

# ==============================================================
# 1️⃣ BUILD TRAJECTORIES FROM BYTE TRACKS
# ==============================================================
def build_trajectories(track_dir):
    trajectories = {}
    frame_to_positions = {}
    npy_files = sorted(Path(track_dir).glob("*.npy"))

    for npy_path in tqdm(npy_files, desc="Building trajectories"):
        try:
            frame_idx = int(npy_path.stem)
        except:
            continue
        dets = np.load(npy_path)
        if dets.size == 0:
            frame_to_positions[frame_idx] = []
            continue

        boxes, tids = dets[:, :4], dets[:, 5].astype(int)
        pairs = []
        for box, tid in zip(boxes, tids):
            pairs.append((tid, box.tolist()))
            if tid not in trajectories:
                trajectories[tid] = []
            trajectories[tid].append({"frame": frame_idx, "bbox": box.tolist()})
        frame_to_positions[frame_idx] = pairs

    return trajectories, frame_to_positions

# ==============================================================
# 2️⃣ BEHAVIORAL FEATURE EXTRACTION
# ==============================================================
def extract_behavioral_features(trajectories, frame_to_positions,
                                window=30, stride=10, radius=50):
    features = []
    for track_id, traj in tqdm(trajectories.items(), desc="Extracting features"):
        if len(traj) < window:
            continue
        traj = sorted(traj, key=lambda x: x['frame'])
        for start in range(0, len(traj)-window+1, stride):
            window_frames = traj[start:start+window]
            centers = [((e['bbox'][0]+e['bbox'][2])/2.0,
                        (e['bbox'][1]+e['bbox'][3])/2.0) for e in window_frames]
            frames = [e['frame'] for e in window_frames]
            densities, interactions = [], []

            for fno, c in zip(frames, centers):
                neigh = [
                    n for tid, n in frame_to_positions.get(fno, [])
                    if tid != track_id and np.linalg.norm(np.array(c)-np.array([(n[0]+n[2])/2, (n[1]+n[3])/2])) < radius
                ]
                densities.append(len(neigh))
                if len(neigh) > 0:
                    dists = [np.linalg.norm(np.array(c)-np.array([(n[0]+n[2])/2, (n[1]+n[3])/2])) for n in neigh]
                    interactions.append(np.mean(dists))

            avg_density = float(np.mean(densities)) if densities else 0.0
            interaction_count = float(np.mean(interactions)) if interactions else 0.0
            center_array = np.array(centers, dtype=np.float32)

            if len(center_array) > 1:
                angles = np.arctan2(center_array[:,1], center_array[:,0])
                dir_var = float(np.var(np.diff(angles)))
                speeds = np.linalg.norm(np.diff(center_array, axis=0), axis=1)
                avg_speed = float(np.mean(speeds)) if speeds.size else 0.0
                std_speed = float(np.std(speeds)) if speeds.size else 0.0
            else:
                dir_var, avg_speed, std_speed = 0.0, 0.0, 0.0

            scatter = float(np.linalg.norm(center_array[-1]-center_array[0])) if len(center_array) > 1 else 0.0

            features.append({
                "track_id": track_id,
                "start_frame": int(frames[0]),
                "end_frame": int(frames[-1]),
                "avg_density": avg_density,
                "interaction_count": interaction_count,
                "direction_variance": dir_var,
                "avg_speed": avg_speed,
                "std_speed": std_speed,
                "scatter": scatter
            })
    return pd.DataFrame(features)

# ==============================================================
# 3️⃣ RISK SCORING + SAVE SUMMARY
# ==============================================================
def get_risk(row):
    return 1 if (
        row['avg_density'] > 1.0 or
        row['interaction_count'] > 30 or
        row['direction_variance'] > 0.0005 or
        row['avg_speed'] > 15 or
        row['std_speed'] > 10 or
        row['scatter'] > 100
    ) else 0

def save_behavioral_features(df):
    df['risk_flag'] = df.apply(get_risk, axis=1)
    csv_path = SAVE_ROOT / "layer3_behavioral_features.csv"
    df.to_csv(csv_path, index=False)
    summary_json = {
        "total_windows": int(len(df)),
        "risk_windows": int(df['risk_flag'].sum()),
        "no_risk_windows": int((df['risk_flag']==0).sum())
    }
    with open(SAVE_ROOT / "layer3_summary.json", "w") as f:
        json.dump(summary_json, f, indent=4)
    return csv_path, summary_json

# ==============================================================
# 4️⃣ MAIN PIPELINE
# ==============================================================
def run_layer3_only():
    trajectories, frame_to_positions = build_trajectories(L1_TRACKS_DIR)
    df_features = extract_behavioral_features(trajectories, frame_to_positions)
    csv_path, summary = save_behavioral_features(df_features)
    print("✅ Layer 3 completed and saved to:", SAVE_ROOT)
    return {"behavioral_csv": str(csv_path), "summary": summary}

# ==============================================================
# RUN
# ==============================================================
results = run_layer3_only()
print(json.dumps(results, indent=4))

✅ Layer3 directory: /content/drive/MyDrive/Store/pipeline_outputs/layer3


Extracting features: 100%|██████████| 444/444 [00:25<00:00, 17.69it/s]


✅ Layer 3 completed and saved to: /content/drive/MyDrive/Store/pipeline_outputs/layer3
{
    "behavioral_csv": "/content/drive/MyDrive/Store/pipeline_outputs/layer3/layer3_behavioral_features.csv",
    "summary": {
        "total_windows": 2328,
        "risk_windows": 1574,
        "no_risk_windows": 754
    }
}


In [ ]:
import pandas as pd
pd.read_csv("/content/drive/MyDrive/Store/pipeline_outputs/layer3/layer3_behavioral_features.csv").head()

,track_id,start_frame,end_frame,avg_density,interaction_count,direction_variance,avg_speed,std_speed,scatter,risk_flag
0,1,1,30,0.0,0.0,0.000923,10.676095,8.851307,30.342182,1
1,1,11,40,0.0,0.0,0.001600,12.640157,12.680813,48.153938,1
2,1,21,50,0.0,0.0,0.002361,14.301537,15.289762,7.364936,1
3,1,31,60,0.0,0.0,0.003177,16.410521,16.891190,7.281336,1
4,1,41,70,0.0,0.0,0.002674,15.336005,15.501948,7.334033,1


# COMPARISON

In [ ]:
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from scipy.spatial.distance import cosine, euclidean
from pathlib import Path
from tqdm import tqdm

# ==============================================================
# CONFIG
# ==============================================================
L2_FEATURES_PATH = Path("/content/drive/MyDrive/Store/pipeline_outputs/layer2/Farneback_features/sequence_flow.npz")
L3_CSV_PATH = Path("/content/drive/MyDrive/Store/pipeline_outputs/layer3/layer3_behavioral_features.csv")
FRAMES_DIR = Path("/content/drive/MyDrive/Store/pipeline_outputs/layer1/frames_for_farneback")
SAVE_ROOT = Path("/content/drive/MyDrive/Store/pipeline_outputs/divergence")

SAVE_ROOT.mkdir(parents=True, exist_ok=True)
print("✅ Divergence output directory:", SAVE_ROOT)

# ==============================================================
# 1️⃣ COMPUTE DIVERGENCE BETWEEN LAYER 2 & LAYER 3
# ==============================================================
def compute_divergence(l2_npz_path, l3_csv_path, save_dir):
    data = np.load(l2_npz_path)
    l2_features = data["features"] if "features" in data else data[list(data.keys())[0]]
    l3_df = pd.read_csv(l3_csv_path)

    l3_features = l3_df[["avg_density","interaction_count","direction_variance",
                         "avg_speed","std_speed","scatter"]].to_numpy()

    n = min(len(l2_features), len(l3_features))
    print(f"➡️ Comparing {n} windows between Layer 2 and Layer 3")

    records = []
    for i in tqdm(range(n), desc="Computing divergence"):
        f2, f3 = np.array(l2_features[i]), np.array(l3_features[i])
        min_len = min(len(f2), len(f3))
        f2, f3 = f2[:min_len], f3[:min_len]

        cos_d = cosine(f2, f3)
        euc_d = euclidean(f2, f3)
        kl_d = np.sum(f2 * np.log((f2 + 1e-8) / (f3 + 1e-8)))
        div_score = (cos_d + euc_d + kl_d) / 3.0
        records.append({
            "window_id": i,
            "cosine": float(cos_d),
            "euclidean": float(euc_d),
            "kl_div": float(kl_d),
            "divergence_score": float(div_score)
        })

    df_div = pd.DataFrame(records)
    csv_path = save_dir / "divergence_summary.csv"
    df_div.to_csv(csv_path, index=False)

    # Plot divergence
    plt.figure(figsize=(10,4))
    plt.plot(df_div["window_id"], df_div["divergence_score"], color="crimson", linewidth=2)
    plt.title("Layer 2 vs Layer 3 Divergence Over Time")
    plt.xlabel("Window ID")
    plt.ylabel("Divergence Score")
    plt.ylim(0, df_div["divergence_score"].max() + 0.05)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / "divergence_plot.png")
    plt.close()

    # Summary JSON
    summary = {
        "num_windows": len(df_div),
        "mean_divergence": float(df_div["divergence_score"].mean()),
        "max_divergence": float(df_div["divergence_score"].max()),
        "min_divergence": float(df_div["divergence_score"].min())
    }
    with open(save_dir / "divergence_summary.json", "w") as f:
        json.dump(summary, f, indent=4)

    print("✅ Divergence computed and saved successfully.")
    return csv_path, summary

# ==============================================================
# 2️⃣ OPTIONAL: VISUALIZE RISK LEVELS ON FRAMES
# ==============================================================
def generate_divergence_video(frames_dir, l3_csv_path, save_dir):
    df = pd.read_csv(l3_csv_path)
    frame_files = sorted(Path(frames_dir).glob("*.jpg"))
    out_path = save_dir / "divergence_visualization.mp4"

    if not frame_files:
        print("⚠️ No frames found for visualization.")
        return None

    sample = cv2.imread(str(frame_files[0]))
    height, width = sample.shape[:2]
    writer = cv2.VideoWriter(str(out_path),
                             cv2.VideoWriter_fourcc(*'mp4v'),
                             15, (width, height))

    print("🎥 Generating divergence visualization...")
    for i, frame_path in enumerate(tqdm(frame_files, desc="Rendering frames")):
        frame = cv2.imread(str(frame_path))
        risk_flag = df.iloc[i % len(df)]['risk_flag']
        color = (0, 0, 255) if risk_flag == 1 else (0, 255, 0)
        cv2.putText(frame, f"Risk: {'HIGH' if risk_flag==1 else 'LOW'}",
                    (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 3)
        cv2.rectangle(frame, (20, 20), (width-20, height-20), color, 5)
        writer.write(frame)
    writer.release()
    print(f"✅ Video saved to {out_path}")
    return out_path

# ==============================================================
# 3️⃣ MAIN WRAPPER
# ==============================================================
def run_divergence_pipeline():
    div_csv, div_summary = compute_divergence(L2_FEATURES_PATH, L3_CSV_PATH, SAVE_ROOT)
    vid_path = generate_divergence_video(FRAMES_DIR, L3_CSV_PATH, SAVE_ROOT)
    final_json = {
        "divergence_csv": str(div_csv),
        "divergence_summary": div_summary,
        "divergence_video": str(vid_path)
    }
    with open(SAVE_ROOT / "divergence_output.json", "w") as f:
        json.dump(final_json, f, indent=4)
    print("\n🎯 Divergence Module Completed! Results stored in:", SAVE_ROOT)
    return final_json

# ==============================================================
# RUN PIPELINE
# ==============================================================
results = run_divergence_pipeline()
print(json.dumps(results, indent=4))

✅ Divergence output directory: /content/drive/MyDrive/Store/pipeline_outputs/divergence
➡️ Comparing 594 windows between Layer 2 and Layer 3


Computing divergence: 100%|██████████| 594/594 [00:00<00:00, 13317.81it/s]


✅ Divergence computed and saved successfully.
🎥 Generating divergence visualization...


Rendering frames: 100%|██████████| 595/595 [00:38<00:00, 15.62it/s]


✅ Video saved to /content/drive/MyDrive/Store/pipeline_outputs/divergence/divergence_visualization.mp4

🎯 Divergence Module Completed! Results stored in: /content/drive/MyDrive/Store/pipeline_outputs/divergence
{
    "divergence_csv": "/content/drive/MyDrive/Store/pipeline_outputs/divergence/divergence_summary.csv",
    "divergence_summary": {
        "num_windows": 594,
        "mean_divergence": 29.518047532597635,
        "max_divergence": 89.04550688233941,
        "min_divergence": 2.676142619810595
    },
    "divergence_video": "/content/drive/MyDrive/Store/pipeline_outputs/divergence/divergence_visualization.mp4"
}


# Visualization


In [ ]:
import cv2, json, numpy as np, matplotlib.pyplot as plt, os
from pathlib import Path

# === Create folders in Drive ===
base_vis_dir = Path("/content/drive/MyDrive/Store/pipeline_visuals")
layer1_dir = base_vis_dir / "layer1_visuals"
layer2_dir = base_vis_dir / "layer2_visuals"
layer3_dir = base_vis_dir / "layer3_visuals"

for d in [layer1_dir, layer2_dir, layer3_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Visualization folders ready at:", base_vis_dir)

# ==============================================================
# 🧩 LAYER 1 — Detection + Tracking Visualization
# ==============================================================

layer1_root = Path("/content/drive/MyDrive/Store/pipeline_outputs/layer1")
frames_dir = layer1_root / "frames_for_farneback"
fused_dir = layer1_root / "fused_dets"
track_dir = layer1_root / "byte_tracks"

frame_paths = sorted(frames_dir.glob("*.jpg"))[:10]

for frame_path in frame_paths:
    img = cv2.imread(str(frame_path))
    H, W = img.shape[:2]

    fused_file = fused_dir / f"{frame_path.stem}.json"
    if fused_file.exists():
        with open(fused_file) as f:
            dets = json.load(f)
        for x1, y1, x2, y2, conf, cls in dets:
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 2)
            cv2.putText(img, f"conf:{conf:.2f}", (int(x1), int(y1)-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

    track_file = track_dir / f"{frame_path.stem}.npy"
    if track_file.exists():
        arr = np.load(track_file)
        for x1, y1, x2, y2, conf, tid in arr:
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0,255,0), 2)
            cv2.putText(img, f"ID:{int(tid)}", (int(x1), int(y1)-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    save_path = layer1_dir / f"{frame_path.stem}_layer1.jpg"
    cv2.imwrite(str(save_path), img)

print(f"✅ Layer 1 visuals saved in: {layer1_dir}")


# ==============================================================
# 🌊 LAYER 2 — Optical Flow Visualization
# ==============================================================

import cv2, numpy as np
from pathlib import Path

layer2_dir = Path("/content/drive/MyDrive/Store/pipeline_visuals/layer2_visuals")
frames_dir = Path("/content/drive/MyDrive/Store/pipeline_outputs/layer1/frames_for_farneback")
frame_files = sorted(frames_dir.glob("*.jpg"))[:11]

# --- Create HSV color wheel legend (for direction) ---
def create_color_wheel(size=150):
    wheel = np.zeros((size, size, 3), dtype=np.uint8)
    center = size // 2
    radius = size // 2 - 2
    for i in range(size):
        for j in range(size):
            dx, dy = j - center, i - center
            mag = np.sqrt(dx*dx + dy*dy)
            if mag > radius:
                continue
            ang = np.arctan2(dy, dx)
            hue = ((ang * 180 / np.pi / 2) + 180) % 180
            val = int(np.clip((mag / radius) * 255, 0, 255))
            wheel[i, j] = (hue, 255, val)
    return cv2.cvtColor(wheel, cv2.COLOR_HSV2BGR)

legend = create_color_wheel(150)

# --- Create speed gradient (for brightness) ---
def create_speed_bar(width=200, height=20):
    bar = np.zeros((height, width, 3), dtype=np.uint8)
    for i in range(width):
        brightness = int((i / width) * 255)
        bar[:, i] = (30, 255, brightness)
    return cv2.cvtColor(bar, cv2.COLOR_HSV2BGR)

speed_bar = create_speed_bar(200, 20)

# --- Generate annotated flow visualizations ---
for i in range(10):
    f1 = cv2.imread(str(frame_files[i]))
    f2 = cv2.imread(str(frame_files[i+1]))
    gray1 = cv2.cvtColor(f1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(f2, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(gray1, gray2, None,
                                        0.5, 3, 15, 3, 5, 1.2, 0)
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros_like(f1)
    hsv[..., 1] = 255
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    flow_rgb = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    # Dimensions
    h, w = flow_rgb.shape[:2]
    lw, lh = legend.shape[1], legend.shape[0]

    # Overlay legend in bottom-right
    flow_rgb[h-lh-10:h-10, w-lw-10:w-10] = legend

    # Resize and place speed bar safely
    max_bar_width = min(w - (w-lw-10), 200)
    sb_resized = cv2.resize(speed_bar, (lw, 20))
    bar_y1, bar_y2 = h - lh - 40, h - lh - 20
    bar_x1, bar_x2 = w - lw - 10, w - lw - 10 + sb_resized.shape[1]
    flow_rgb[bar_y1:bar_y2, bar_x1:bar_x2] = sb_resized

    # --- Add annotation text ---
    cv2.putText(flow_rgb, "Optical Flow: Motion Direction & Speed",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    cv2.putText(flow_rgb, "Color = Direction  |  Brightness = Speed",
                (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    # Direction labels near legend
    cv2.putText(flow_rgb, " Green: Left Motion", (w-lw-260, h-40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
    cv2.putText(flow_rgb, " Red: Right Motion", (w-lw-260, h-60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 2)
    cv2.putText(flow_rgb, " Blue: Upward", (w-lw-260, h-80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 2)
    cv2.putText(flow_rgb, " Yellow: Downward", (w-lw-260, h-100),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 2)

    # Speed bar labels
    cv2.putText(flow_rgb, "Slow", (bar_x1 - 45, bar_y2 - 3),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
    cv2.putText(flow_rgb, "Fast", (bar_x2 + 5, bar_y2 - 3),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)

    out_path = layer2_dir / f"{frame_files[i].stem}_layer2_flow_annotated.jpg"
    cv2.imwrite(str(out_path), flow_rgb)

print("✅ Saved all annotated Layer 2 flow visuals with labeled color + speed legends in:", layer2_dir)

# ==============================================================
# 🚶‍♂️ LAYER 3 — Behavioral Trajectory Visualization
# ==============================================================


import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
import networkx as nx

# === Directories ===
layer1_tracks = Path("/content/drive/MyDrive/Store/pipeline_outputs/layer1/byte_tracks")
frames_dir = Path("/content/drive/MyDrive/Store/pipeline_outputs/layer1/frames_for_farneback")
save_dir = Path("/content/drive/MyDrive/Store/pipeline_visuals/layer3_gcn_visuals")
save_dir.mkdir(parents=True, exist_ok=True)

# === Parameters ===
frame_sample = sorted(frames_dir.glob("*.jpg"))[:10]  # first 10 frames
distance_threshold = 80  # pixel distance for edge creation
node_size = 200
edge_color = (150, 150, 150)
node_color_high_density = (255, 100, 100)
node_color_normal = (0, 255, 0)

def get_centroids(track_file):
    """Extract person centroids from ByteTrack file"""
    arr = np.load(track_file)
    if arr.size == 0:
        return []
    centroids = []
    for x1, y1, x2, y2, conf, tid in arr:
        cx, cy = (x1+x2)/2, (y1+y2)/2
        centroids.append((int(cx), int(cy), int(tid)))
    return centroids

# === Visualization Loop ===
for frame_path in frame_sample:
    img = cv2.imread(str(frame_path))
    H, W = img.shape[:2]

    track_file = layer1_tracks / f"{frame_path.stem}.npy"
    if not track_file.exists():
        continue

    centroids = get_centroids(track_file)
    if len(centroids) == 0:
        continue

    # Build graph
    G = nx.Graph()
    for i, (cx, cy, tid) in enumerate(centroids):
        G.add_node(tid, pos=(cx, cy))
    for i, (cx1, cy1, tid1) in enumerate(centroids):
        for j, (cx2, cy2, tid2) in enumerate(centroids):
            if i < j:
                dist = np.sqrt((cx1-cx2)**2 + (cy1-cy2)**2)
                if dist < distance_threshold:
                    G.add_edge(tid1, tid2, weight=1.0)

    # Draw edges (as light gray lines)
    for (u, v) in G.edges():
        p1 = G.nodes[u]['pos']
        p2 = G.nodes[v]['pos']
        cv2.line(img, (int(p1[0]), int(p1[1])), (int(p2[0]), int(p2[1])), edge_color, 1)

    # Draw nodes (as circles)
    for tid, data in G.nodes(data=True):
        cx, cy = data['pos']
        # Color nodes based on degree (number of edges)
        deg = G.degree(tid)
        color = node_color_high_density if deg > 3 else node_color_normal
        cv2.circle(img, (int(cx), int(cy)), 6, color, -1)
        cv2.putText(img, f"{tid}", (cx+6, cy-6), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)

    # Overlay title
    cv2.putText(img, "Layer 3 — GCN Graph Construction (Nodes = People, Edges = Proximity)",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    cv2.putText(img, f"Edges < {distance_threshold}px = Interaction Links",
                (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)

    # Save output
    out_path = save_dir / f"{frame_path.stem}_gcn_graph.jpg"
    cv2.imwrite(str(out_path), img)

print(f"✅ GCN-style graph visualizations saved to: {save_dir}")


# ==============================================================
# ✅ VERIFY ALL SAVED FILES
# ==============================================================

!echo "Layer 1:" && ls -lh "/content/drive/MyDrive/Store/pipeline_visuals/layer1_visuals" | head
!echo "Layer 2:" && ls -lh "/content/drive/MyDrive/Store/pipeline_visuals/layer2_visuals" | head
!echo "Layer 3:" && ls -lh "/content/drive/MyDrive/Store/pipeline_visuals/layer3_visuals" | head


✅ Visualization folders ready at: /content/drive/MyDrive/Store/pipeline_visuals
✅ Layer 1 visuals saved in: /content/drive/MyDrive/Store/pipeline_visuals/layer1_visuals
✅ Saved all annotated Layer 2 flow visuals with labeled color + speed legends in: /content/drive/MyDrive/Store/pipeline_visuals/layer2_visuals
✅ GCN-style graph visualizations saved to: /content/drive/MyDrive/Store/pipeline_visuals/layer3_gcn_visuals
Layer 1:
total 6.6M
-rw------- 1 root root 654K Nov 13 15:06 000001_layer1.jpg
-rw------- 1 root root 643K Nov 13 15:06 000002_layer1.jpg
-rw------- 1 root root 648K Nov 13 15:06 000003_layer1.jpg
-rw------- 1 root root 659K Nov 13 15:06 000004_layer1.jpg
-rw------- 1 root root 667K Nov 13 15:06 000005_layer1.jpg
-rw------- 1 root root 664K Nov 13 15:06 000006_layer1.jpg
-rw------- 1 root root 680K Nov 13 15:06 000007_layer1.jpg
-rw------- 1 root root 688K Nov 13 15:06 000008_layer1.jpg
-rw------- 1 root root 675K Nov 13 15:06 000009_layer1.jpg
Layer 2:
total 7.2M
-rw------

In [2]:
import imageio, glob
import cv2, json, numpy as np, matplotlib.pyplot as plt, os
from pathlib import Path

# === Path setup ===
layer1_dir = "/content/drive/MyDrive/Store/pipeline_visuals/layer1_visuals"
gif_path = os.path.join(layer1_dir, "layer1_summary.gif")

# === Load the saved frames (sorted by filename) ===
frames = sorted(glob.glob(os.path.join(layer1_dir, "*_layer1.jpg")))

# === Create GIF ===
images = [imageio.imread(f) for f in frames]
imageio.mimsave(gif_path, images, fps=2)  # slower = 2fps, faster = 5fps

print(f"✅ GIF created at: {gif_path}")


/tmp/ipython-input-4219310681.py:13: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images = [imageio.imread(f) for f in frames]


✅ GIF created at: /content/drive/MyDrive/Store/pipeline_visuals/layer1_visuals/layer1_summary.gif
